In [ ]:
# pip install fastapi uvicorn ollama openai python-multipart python-dotenv mysql-connector-python requests

import os
import base64
import uvicorn
import nest_asyncio
from fastapi import FastAPI, Request, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from openai import OpenAI
import ollama
from dotenv import load_dotenv
import database

# 환경 변수 로드
load_dotenv()

# Jupyter 환경에서 FastAPI 실행을 위한 설정
nest_asyncio.apply()

app = FastAPI()

# CORS 설정: 모든 Origin/Method/Header 허용
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def saveAnalysisResult(userQuestion, aiResponse, imagePath):
    """ 분석 결과를 MySQL 데이터베이스에 저장합니다. """
    try:
        query = "INSERT INTO analysis_history (question, response, imagePath) VALUES (%s, %s, %s)"
        params = (userQuestion, aiResponse, imagePath)
        result = database.executeQuery(query, params)
        return result
    except Exception as e:
        print(f"Save error: {e}")
        return {"success": False, "message": str(e)}

async def processWithGpt(imageContent, userQuestion):
    """ OpenAI GPT 모델을 사용하여 이미지를 분석합니다. """
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    base64Image = base64.b64encode(imageContent).decode('utf-8')
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": userQuestion},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64Image}"}
                    }
                ]
            }
        ]
    )
    return response.choices[0].message.content

async def processWithOllama(imageContent, userQuestion):
    """ Ollama 로컬 모델을 사용하여 이미지를 분석합니다. """
    modelName = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
    
    response = ollama.chat(
        model=modelName,
        messages=[{
            'role': 'user',
            'content': userQuestion,
            'images': [imageContent]
        }]
    )
    return response['message']['content']

@app.post("/analyze")
async def analyzeEndpoint(file: UploadFile = File(...), question: str = Form(...)):
    """ 이미지 파일을 업로드받아 텍스트를 추출하고 질문에 답변합니다. """
    try:
        # 파일 저장 경로 설정 (dataset 폴더 활용)
        uploadDir = "dataset"
        if not os.path.exists(uploadDir):
            os.makedirs(uploadDir)
            
        filePath = os.path.join(uploadDir, file.filename)
        fileContent = await file.read()
        
        with open(filePath, "wb") as f:
            f.write(fileContent)
            
        # 모델 스위칭 로직
        useModel = os.getenv("USE_MODEL", "OLLAMA")
        aiResponse = ""
        
        if useModel == "GPT":
            aiResponse = await processWithGpt(fileContent, question)
        elif useModel == "OLLAMA":
            aiResponse = await processWithOllama(fileContent, question)
        else:
            aiResponse = "지원하지 않는 모델 설정입니다."
            
        # 데이터베이스 저장
        saveAnalysisResult(question, aiResponse, filePath)
        
        return {"success": True, "data": aiResponse}
        
    except Exception as e:
        # 에러 발생 시 지정된 JSON 형식 반환
        return JSONResponse(
            status_code=500,
            content={"success": False, "message": str(e)}
        )

if __name__ == "__main__":
    # 서버 실행 (Port: 8000)
    print("Starting Image Analysis Server...")
    uvicorn.run(app, host="0.0.0.0", port=8000)